# Oracle DB 실시간 조회 대시보드 
- Gradio 화면에서 사용자가 카테고리를 선택하면, 그 즉시 **Oracle DB에 쿼리를 날려서** 결과
- Chapter02(Oracle 연동)와 Chapter03(Gradio)을 하나로 합치는 "미니 종합 실습"

In [11]:
import os
from dotenv import load_dotenv
import oracledb
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib as mpl
import gradio as gr

mpl.rc('font', family='AppleGothic')
mpl.rc('axes', unicode_minus=False)

load_dotenv()

USER = os.getenv("ORACLE_USER")
PASSWORD = os.getenv("ORACLE_PASSWORD")
DSN = os.getenv("ORACLE_DSN")


In [12]:
def query_category(category):

    # 사용자가 입력값을 직접 SQL 문자열에 끼워넣지 않도록
    # :category 바인드 변수로 안전하게 사용(SQL INJECTION 방지)
    query = "SELECT * FROM DELIVERY_ORDERS WHERE CATEGORY = :category"
    with oracledb.connect(user=USER, password=PASSWORD, dsn=DSN) as conn:
        df = pd.read_sql(query, conn, params={"category": category})

    if len(df) == 0:
        # 데이터가 없는 경우를 대비한 방어 코드
        empty_df = pd.DataFrame({"안내": ["해당 카테고리의 데이터가 없습니다."]})
        fig, ax = plt.subplots()
        ax.text(0.5,0.5, "데이터 없음", ha="center")
        return empty_df, fig
    # 요약 통계
    summary = df[["PRICE","RATING"]].describe().round(1)
    
    # 가격 히스토그램
    fig, ax = plt.subplots(figsize=(6, 4))
    ax.hist(df["PRICE"], bins=8, color="salmon", edgecolor="black")
    ax.set_title(f"'{category}' 카테고리 가격 분포 (n={len(df)}건)")
    ax.set_xlabel("가격")
    ax.set_ylabel("건수")

    return summary, fig


In [ ]:
with gr.Blocks(title="배달 주문 실시간 대시보드") as demo:
    gr.Markdown("##  카테고리별 주문 데이터 실시간 조회")
    gr.Markdown("카테고리를 선택하면 Oracle DB에서 즉시 데이터를 조회합니다.")

    with gr.Row():
        category_dropdown = gr.Dropdown(
            ["분식", "치킨", "피자", "중식", "일식"],
            label="카테고리 선택",
        )
        search_btn = gr.Button("조회하기")

    with gr.Row():
        summary_output = gr.Dataframe(label="요약 통계")
        plot_output = gr.Plot(label="가격 분포")

    search_btn.click(
        fn=query_category,
        inputs=category_dropdown,
        outputs=[summary_output, plot_output],
    )

demo.launch()

* Running on local URL:  http://127.0.0.1:7863
* To create a public link, set `share=True` in `launch()`.


/var/folders/4j/qgl7j0hx7y33d7l5bjgf9h9h0000gn/T/ipykernel_27877/1299831598.py:7: UserWarning: pandas only supports SQLAlchemy connectable (engine/connection) or database string URI or sqlite3 DBAPI2 connection. Other DBAPI2 objects are not tested. Please consider using SQLAlchemy.
  df = pd.read_sql(query, conn, params={"category": category})
/var/folders/4j/qgl7j0hx7y33d7l5bjgf9h9h0000gn/T/ipykernel_27877/1299831598.py:7: UserWarning: pandas only supports SQLAlchemy connectable (engine/connection) or database string URI or sqlite3 DBAPI2 connection. Other DBAPI2 objects are not tested. Please consider using SQLAlchemy.
  df = pd.read_sql(query, conn, params={"category": category})
/var/folders/4j/qgl7j0hx7y33d7l5bjgf9h9h0000gn/T/ipykernel_27877/1299831598.py:7: UserWarning: pandas only supports SQLAlchemy connectable (engine/connection) or database string URI or sqlite3 DBAPI2 connection. Other DBAPI2 objects are not tested. Please consider using SQLAlchemy.
  df = pd.read_sql(query,

In [14]:
demo.close()


Closing server running on port: 7863
